# Chapter 7: More on Bracket Algebra

**Source orientation.** Sections 7.1-7.5; printed pages 109-126; PDF pages 131-148. The assigned source span is used for topic coverage: determinant vectors, reconstruction from brackets, projective invariants, relative weights, rational invariant functions, the bracket ring, Grassmann-Pluecker relations, and the two fundamental-theorem viewpoints. The prose, diagrams, code, and checks below are original.

## Chapter Question

What does a projective configuration look like if the brackets, rather than the point coordinates, are treated as the primary data?

The chapter route is computational: build the determinant vector `Gamma(P)`, normalize enough brackets to reconstruct a basis, track the weights that make rational functions invariant, and use symbolic expansion to see how bracket algebra compresses coordinate calculations.

In [ ]:
from pathlib import Path
import itertools
import math
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from matplotlib.patches import FancyArrowPatch

from utils.artifacts import (
    assert_artifacts,
    book_relative,
    display_artifact,
    image_stats,
    save_json,
    save_table,
)
from utils.projective import affine

CHAPTER_SLUG = "chapter-07-more-on-bracket-algebra"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / CHAPTER_SLUG
FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
TABLES = ARTIFACT_ROOT / "tables"
CHECKS = ARTIFACT_ROOT / "checks"
for folder in (FIGURES, HTML, TABLES, CHECKS):
    folder.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 140, "font.size": 10})
checks = {}
artifact_paths = {}
POINTS = tuple(range(1, 7))
TRIPLES = list(itertools.combinations(POINTS, 3))


def save_current_figure(path, *, tight=True):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, bbox_inches="tight" if tight else None, facecolor="white")
    plt.close()
    return path


def col(P, i):
    return np.asarray(P)[:, i - 1]


def det_bracket(P, triple):
    return float(np.linalg.det(np.column_stack([col(P, i) for i in triple])))


def gamma(P):
    return {triple: det_bracket(P, triple) for triple in TRIPLES}


def permutation_sign(items):
    inversions = 0
    data = list(items)
    for a in range(len(data)):
        for b in range(a + 1, len(data)):
            inversions += int(data[a] > data[b])
    return -1 if inversions % 2 else 1


def alternating_lookup(G, triple):
    if len(set(triple)) < len(triple):
        return 0.0
    ordered = tuple(sorted(triple))
    return permutation_sign(triple) * G[ordered]


def gp_residual_from_gamma(G, a, b, c, d, e, f):
    return (
        alternating_lookup(G, (a, b, c)) * alternating_lookup(G, (d, e, f))
        - alternating_lookup(G, (a, b, d)) * alternating_lookup(G, (c, e, f))
        + alternating_lookup(G, (a, b, e)) * alternating_lookup(G, (c, d, f))
        - alternating_lookup(G, (a, b, f)) * alternating_lookup(G, (c, d, e))
    )


def monomial_weight(factors, n=6):
    weight = {i: 0 for i in range(1, n + 1)}
    for triple in factors:
        for i in triple:
            weight[i] += 1
    return weight


def monomial_value(P, factors):
    value = 1.0
    for triple in factors:
        value *= det_bracket(P, triple)
    return value


def bracket_quotient(P, numerator, denominator):
    return monomial_value(P, numerator) / monomial_value(P, denominator)


def reconstruct_from_normalized_gamma(G):
    P = np.zeros((3, 6), dtype=float)
    P[:, :3] = np.eye(3)
    for i in range(4, 7):
        P[:, i - 1] = [
            alternating_lookup(G, (2, 3, i)),
            -alternating_lookup(G, (1, 3, i)),
            alternating_lookup(G, (1, 2, i)),
        ]
    return P


def as_book_path(path):
    return book_relative(path)


## Computational Translation

| Source idea | Computational object | What to inspect |
| --- | --- | --- |
| A labeled point configuration | A `3 x n` matrix `P` whose columns are homogeneous vectors | Columns may be rescaled; projective transformations multiply all brackets by a common determinant factor. |
| A bracket `[i,j,k]` | The determinant of columns `i`, `j`, and `k` | Sign changes under odd permutations; repeated letters give zero; zero detects collinearity. |
| The determinant vector `Gamma(P)` | The list of all sorted `3 x 3` minors | It behaves like projective coordinates for the whole configuration. |
| Grassmann-Pluecker relations | Quadratic relations among bracket symbols | They are the consistency checks that distinguish real determinant data from arbitrary numbers. |
| Projectively invariant functions | Balanced quotients of bracket monomials | Matching point weights in numerator and denominator makes all column scalings cancel. |
| Bracket algebra | A quotient ring in bracket variables | Repeat, alternation, and Grassmann-Pluecker rules are treated as algebraic rewrite rules. |

**Library routing.** Matplotlib is used for durable 2D determinant and reconstruction diagrams; Plotly is used for the scaling lab because the invariant is best seen under parameter motion; NetworkX is used for proof and cancellation dependencies; SymPy is used for exact bracket identities and coordinate expansion counts; Pandas/CSV ledgers record the data behind the visuals.

## Visualization Planner Pass

The source-specific storyboard below is the plan implemented by the notebook cells.

1. **Points-to-determinants map.** Show a six-point configuration, a projective image, and the common determinant scaling of `Gamma(P)`. Validation: every nonzero bracket of `H P` equals `det(H)` times the original bracket.
2. **Basis reconstruction from brackets.** Normalize `[1,2,3]` to one, fill a matrix from `[2,3,i]`, `-[1,3,i]`, and `[1,2,i]`, then verify all other brackets. Validation: the reconstructed determinant vector equals the normalized determinant vector.
3. **Monomial weight balance.** Compare the point weights of a cross-ratio quotient and the two conic bracket monomials. Validation: numerator and denominator weights match exactly.
4. **Bracket polynomial cancellation and proof graph.** Expand the conic bracket expression and count raw terms, cancellations, and survivors. Validation: symbolic Grassmann-Pluecker expansion is zero and the conic expression vanishes on sampled conic points.
5. **Applied scaling lab.** Vary one homogeneous column scale and compare a balanced quotient with an intentionally unbalanced expression. Validation: the balanced quotient is flat while the unbalanced expression changes.

The artifacts use concept names so a stale or generic render is easy to spot.

In [ ]:
storyboard = {
    "chapter_goal": "Use bracket values as primary coordinates for projective configurations and invariants.",
    "source_span_read": "printed pages 109-126 / PDF pages 131-148",
    "concept_inventory": [
        "Gamma(P) as the vector of all 3x3 determinants",
        "common GL(3) determinant scaling and per-point homogeneous scaling",
        "Grassmann-Pluecker consistency relations",
        "basis reconstruction after normalizing one nonzero bracket",
        "functional basis viewpoint for projective invariants",
        "balanced rational bracket functions",
        "bracket ring modulo repeat, alternation, and Grassmann-Pluecker ideals",
        "coordinate expansion compression and cancellation",
    ],
    "library_routing": [
        {"concept": "determinant vector", "library": "Matplotlib", "reason": "labeled static comparison and residual scatter"},
        {"concept": "basis reconstruction", "library": "Matplotlib + CSV", "reason": "matrix entries and residual ledger are inspectable"},
        {"concept": "monomial weights", "library": "Matplotlib + Pandas", "reason": "weight balance is a small table/heatmap"},
        {"concept": "cancellation proof graph", "library": "NetworkX + SymPy", "reason": "dependencies and exact expansion counts are proof data"},
        {"concept": "scaling lab", "library": "Plotly", "reason": "parameter sweep exposes invariant versus non-invariant behavior"},
    ],
    "visual_sequence": [
        "figures/points-to-determinants-gamma-map.png",
        "figures/basis-reconstruction-from-brackets.png",
        "figures/monomial-weight-balance.png",
        "figures/bracket-polynomial-cancellation-proof-graph.png",
        "html/bracket-weight-invariant-lab.html",
    ],
    "computational_checks": [
        "Gamma(HP) common-scale residual",
        "numeric Grassmann-Pluecker residual",
        "basis reconstruction residual over all triples",
        "balanced monomial weight equality",
        "cross-ratio quotient invariance under GL(3) and column scalings",
        "symbolic repeat, alternation, and Grassmann-Pluecker identities",
        "conic bracket expression vanishes on sampled conic points and changes after perturbation",
        "artifact existence, size, and raster nonblank checks",
    ],
    "risks": [
        "symbolic expansion is intentionally small: six points in RP2 only",
        "Plotly HTML is saved as a standalone artifact for static notebook exports",
    ],
}
storyboard_path = save_json(storyboard, ARTIFACT_ROOT, "checks", "storyboard.json")
{"storyboard": as_book_path(storyboard_path), "visuals_planned": len(storyboard["visual_sequence"])}


## 1. From Points To The Determinant Vector

For six labeled points in the projective plane, `Gamma(P)` is the vector of all twenty sorted brackets. A projective transformation `H` sends every bracket to `det(H)` times its old value. This is the first reason brackets are useful: individual coordinates change, but the full determinant vector changes predictably.

The left panel below shows an original affine chart and a projective image of the same six labels. The right panel plots each transformed bracket after division by `det(H)`. If brackets are carrying the projective data correctly, every point in that scatter should land on the diagonal.

In [ ]:
affine_points = np.array([
    [-1.35, -0.55],
    [1.10, -0.48],
    [-1.05, 0.95],
    [0.74, 1.12],
    [0.02, 0.18],
    [1.42, 0.72],
])
P = np.vstack([affine_points.T, np.ones(6)])
H = np.array([
    [1.18, 0.28, 0.32],
    [-0.18, 1.06, 0.24],
    [0.10, -0.07, 1.00],
])
Q = H @ P
det_H = float(np.linalg.det(H))
G_P = gamma(P)
G_Q = gamma(Q)
scaled_residuals = {
    triple: G_Q[triple] / det_H - G_P[triple]
    for triple in TRIPLES
}
checks["gamma_common_scale_residual"] = float(max(abs(v) for v in scaled_residuals.values()))
checks["numeric_gp_residual"] = float(abs(gp_residual_from_gamma(G_P, 1, 2, 3, 4, 5, 6)))

rows = []
for triple in TRIPLES:
    rows.append({
        "triple": "".join(map(str, triple)),
        "bracket_original": G_P[triple],
        "bracket_projective_image": G_Q[triple],
        "image_div_detH": G_Q[triple] / det_H,
        "scale_residual": scaled_residuals[triple],
    })
artifact_paths["gamma_table"] = save_table(rows, ARTIFACT_ROOT, "tables", "gamma-determinant-vector.csv")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
ax = axes[0]
original_xy = np.array([affine(col(P, i)) for i in POINTS])
image_xy = np.array([affine(col(Q, i)) for i in POINTS])
ax.scatter(original_xy[:, 0], original_xy[:, 1], s=58, color="#1f77b4", label="P")
ax.scatter(image_xy[:, 0], image_xy[:, 1], s=58, color="#d62728", marker="s", label="H P")
for i, (a, b) in enumerate(zip(original_xy, image_xy), start=1):
    ax.annotate(str(i), a + np.array([0.035, 0.035]), color="#1f77b4", weight="bold")
    ax.annotate(str(i), b + np.array([0.035, -0.085]), color="#d62728", weight="bold")
    ax.add_patch(FancyArrowPatch(a, b, arrowstyle="->", mutation_scale=9, lw=0.8, color="#6b7280", alpha=0.65))
ax.axhline(0, color="#d1d5db", lw=0.8)
ax.axvline(0, color="#d1d5db", lw=0.8)
ax.set_aspect("equal", adjustable="datalim")
ax.set_title("labeled points before and after H")
ax.legend(loc="upper left", frameon=False)

ax = axes[1]
x = np.array([G_P[t] for t in TRIPLES])
y = np.array([G_Q[t] / det_H for t in TRIPLES])
ax.scatter(x, y, color="#0f766e", s=45)
lo = min(x.min(), y.min()) - 0.5
hi = max(x.max(), y.max()) + 0.5
ax.plot([lo, hi], [lo, hi], color="#111827", lw=1.2, label="expected: y=x")
for triple, xi, yi in zip(TRIPLES, x, y):
    if abs(xi) == max(abs(x)) or triple in [(1, 2, 3), (4, 5, 6)]:
        ax.annotate("".join(map(str, triple)), (xi, yi), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set_xlabel("[i,j,k] for P")
ax.set_ylabel("[i,j,k] for H P divided by det(H)")
ax.set_title("Gamma(P) changes by one common scale")
ax.legend(frameon=False)
fig.suptitle("Points-to-determinants pipeline", y=1.02, fontsize=13)
artifact_paths["gamma_map"] = save_current_figure(FIGURES / "points-to-determinants-gamma-map.png")

{"det_H": det_H, "max_scale_residual": checks["gamma_common_scale_residual"], "gp_residual": checks["numeric_gp_residual"]}


## 2. Reconstructing A Basis From Brackets

Now reverse the direction. If one bracket is nonzero, relabel and rescale the determinant data so `[1,2,3]=1`. In that normalized gauge, the first three columns can be chosen as the standard basis. Every later column is then read directly from brackets:

`column i = ([2,3,i], -[1,3,i], [1,2,i])` for `i >= 4`.

The point of the Grassmann-Pluecker relations is that once these entries are fixed, all remaining brackets are forced. The visual shows the reconstructed matrix and a residual check over every triple, not just the brackets used to fill the matrix.

In [ ]:
base = G_P[(1, 2, 3)]
if abs(base) < 1e-12:
    raise RuntimeError("The chosen base bracket [1,2,3] is zero; relabel the example.")
G_normalized = {triple: value / base for triple, value in G_P.items()}
P_reconstructed = reconstruct_from_normalized_gamma(G_normalized)
G_reconstructed = gamma(P_reconstructed)
reconstruction_residuals = {triple: G_reconstructed[triple] - G_normalized[triple] for triple in TRIPLES}
checks["basis_reconstruction_max_residual"] = float(max(abs(v) for v in reconstruction_residuals.values()))

ledger = []
for i in range(4, 7):
    ledger.append({
        "entry_type": "coordinate",
        "point": i,
        "triple": "",
        "x_from_[23i]": alternating_lookup(G_normalized, (2, 3, i)),
        "y_from_-[13i]": -alternating_lookup(G_normalized, (1, 3, i)),
        "z_from_[12i]": alternating_lookup(G_normalized, (1, 2, i)),
        "normalized_bracket": "",
        "reconstructed_bracket": "",
        "residual": "",
    })
for triple in TRIPLES:
    ledger.append({
        "entry_type": "bracket_residual",
        "point": "",
        "triple": "".join(map(str, triple)),
        "x_from_[23i]": "",
        "y_from_-[13i]": "",
        "z_from_[12i]": "",
        "normalized_bracket": G_normalized[triple],
        "reconstructed_bracket": G_reconstructed[triple],
        "residual": reconstruction_residuals[triple],
    })
artifact_paths["reconstruction_table"] = save_table(ledger, ARTIFACT_ROOT, "tables", "basis-reconstruction-ledger.csv")

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.8))
ax = axes[0]
im = ax.imshow(P_reconstructed, cmap="PuBuGn", aspect="auto")
ax.set_xticks(range(6), labels=[str(i) for i in POINTS])
ax.set_yticks(range(3), labels=["x", "y", "z"])
for r in range(3):
    for c in range(6):
        ax.text(c, r, f"{P_reconstructed[r, c]:.2f}", ha="center", va="center", fontsize=9, color="#111827")
for c in range(3, 6):
    ax.text(c, -0.55, f"from brackets\nwith {c+1}", ha="center", va="center", fontsize=8, color="#374151")
ax.set_title("normalized reconstruction matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[1]
res = np.array([reconstruction_residuals[t] for t in TRIPLES])
ax.bar(range(len(TRIPLES)), res, color="#7c3aed")
ax.axhline(0, color="#111827", lw=0.9)
ax.set_xticks(range(len(TRIPLES)), labels=["".join(map(str, t)) for t in TRIPLES], rotation=70)
ax.set_ylabel("reconstructed minus normalized bracket")
ax.set_title("all bracket residuals after reconstruction")
ax.set_ylim(-1e-12, 1e-12)
fig.suptitle("Basis reconstruction from bracket data", y=1.02, fontsize=13)
artifact_paths["basis_reconstruction"] = save_current_figure(FIGURES / "basis-reconstruction-from-brackets.png")

{"base_bracket": base, "max_reconstruction_residual": checks["basis_reconstruction_max_residual"]}


## 3. Invariant Functions As Balanced Bracket Quotients

A bracket transforms in two ways under `P -> H P D`: it receives a common factor `det(H)` and one column-scale factor for each letter inside the bracket. A quotient of bracket monomials is projectively invariant when the numerator and denominator have the same bracket degree and the same letter weights.

The cross-ratio of four collinear points can be written with an off-line anchor point `e` as a balanced bracket quotient:

`[a,c,e][b,d,e] / ([a,d,e][b,c,e])`.

The same weight idea explains why the two monomials in the six-point conic bracket test can be compared: each point appears twice in each monomial.

In [ ]:
CROSS_NUM = [(1, 3, 5), (2, 4, 5)]
CROSS_DEN = [(1, 4, 5), (2, 3, 5)]
CONIC_A = [(1, 2, 3), (1, 5, 6), (4, 2, 6), (4, 5, 3)]
CONIC_B = [(4, 5, 6), (4, 2, 3), (1, 5, 3), (1, 2, 6)]

line_config = np.array([
    [-1.30, -0.20, 0.85, 1.90, 0.20, 0.75],
    [0.00, 0.00, 0.00, 0.00, 1.15, -0.55],
    [1.00, 1.00, 1.00, 1.00, 1.00, 1.00],
])
column_scales = np.diag([1.7, 0.6, 1.25, 2.1, 0.75, 1.4])
line_image = H @ line_config @ column_scales
cross_original = bracket_quotient(line_config, CROSS_NUM, CROSS_DEN)
cross_image = bracket_quotient(line_image, CROSS_NUM, CROSS_DEN)
checks["cross_ratio_error"] = float(abs(cross_original - cross_image))

weight_rows = []
for label, factors in [
    ("cross-ratio numerator", CROSS_NUM),
    ("cross-ratio denominator", CROSS_DEN),
    ("conic monomial A", CONIC_A),
    ("conic monomial B", CONIC_B),
]:
    weights = monomial_weight(factors)
    row = {"monomial": label, "bracket_degree": len(factors), "factors": " ".join("[" + ",".join(map(str, t)) + "]" for t in factors), "weight_signature": ",".join(str(weights[i]) for i in POINTS)}
    row.update({f"point_{i}": weights[i] for i in POINTS})
    weight_rows.append(row)
artifact_paths["weight_table"] = save_table(weight_rows, ARTIFACT_ROOT, "tables", "monomial-weight-ledger.csv")

cross_weight_ok = monomial_weight(CROSS_NUM) == monomial_weight(CROSS_DEN)
conic_weight_ok = monomial_weight(CONIC_A) == monomial_weight(CONIC_B)
checks["cross_ratio_weights_balanced"] = bool(cross_weight_ok)
checks["conic_monomial_weights_balanced"] = bool(conic_weight_ok)

weight_frame = pd.DataFrame(weight_rows).set_index("monomial")[[f"point_{i}" for i in POINTS]]
fig, ax = plt.subplots(figsize=(9.0, 4.6))
im = ax.imshow(weight_frame.to_numpy(), cmap="YlGnBu", vmin=0, vmax=2)
ax.set_xticks(range(6), labels=[str(i) for i in POINTS])
ax.set_yticks(range(len(weight_frame.index)), labels=weight_frame.index)
for r in range(weight_frame.shape[0]):
    for c in range(weight_frame.shape[1]):
        ax.text(c, r, str(int(weight_frame.iloc[r, c])), ha="center", va="center", color="#111827", weight="bold")
ax.set_xlabel("point label")
ax.set_title("Monomial weights: equal rows mean scalings cancel")
fig.colorbar(im, ax=ax, fraction=0.035, pad=0.04, label="letter count")
artifact_paths["weight_balance"] = save_current_figure(FIGURES / "monomial-weight-balance.png")

{"cross_ratio_original": cross_original, "cross_ratio_image": cross_image, "cross_ratio_error": checks["cross_ratio_error"]}


## 4. Bracket Algebra And Cancellation Graphs

The bracket ring makes three determinant facts into algebraic rules: repeated letters vanish, odd permutations change sign, and Grassmann-Pluecker quadratics vanish. The exact symbolic checks below expand bracket symbols back to coordinate determinants and verify these rules under the expansion map.

The conic bracket expression is a useful stress test. It is short at bracket level but expands into many coordinate monomials. The graph records the proof state: balanced bracket monomials, determinant expansion, opposite-term cancellations, and the surviving coordinate polynomial that tests six points on a conic.

In [ ]:
symbols = {i: sp.symbols(f"x{i} y{i} z{i}") for i in POINTS}
all_symbols = [s for i in POINTS for s in symbols[i]]


def sym_bracket(triple):
    return sp.Matrix.hstack(*[sp.Matrix(symbols[i]) for i in triple]).det()


repeat_identity = sp.simplify(sym_bracket((1, 1, 2)))
alternation_identity = sp.simplify(sym_bracket((1, 2, 3)) + sym_bracket((1, 3, 2)))
gp_identity = sp.simplify(
    sym_bracket((1, 2, 3)) * sym_bracket((4, 5, 6))
    - sym_bracket((1, 2, 4)) * sym_bracket((3, 5, 6))
    + sym_bracket((1, 2, 5)) * sym_bracket((3, 4, 6))
    - sym_bracket((1, 2, 6)) * sym_bracket((3, 4, 5))
)
checks["symbolic_repeat_zero"] = bool(repeat_identity == 0)
checks["symbolic_alternation_zero"] = bool(alternation_identity == 0)
checks["symbolic_gp_zero"] = bool(gp_identity == 0)

conic_expr = (
    math.prod([sym_bracket(t) for t in CONIC_A])
    - math.prod([sym_bracket(t) for t in CONIC_B])
)
conic_expanded = sp.expand(conic_expr)
conic_poly = sp.Poly(conic_expanded, all_symbols)
raw_coordinate_terms = 2 * (math.factorial(3) ** 4)
surviving_terms = len(conic_poly.terms())
cancelled_terms = raw_coordinate_terms - surviving_terms
checks["conic_raw_coordinate_terms"] = int(raw_coordinate_terms)
checks["conic_surviving_coordinate_terms"] = int(surviving_terms)
checks["conic_cancelled_coordinate_terms"] = int(cancelled_terms)


def conic_point(t):
    return np.array([1.0, t, t * t])


conic_params = np.array([-1.8, -0.9, -0.25, 0.55, 1.15, 1.75])
conic_P = np.column_stack([conic_point(t) for t in conic_params])
conic_value = monomial_value(conic_P, CONIC_A) - monomial_value(conic_P, CONIC_B)
perturbed_P = conic_P.copy()
perturbed_P[:, 5] += np.array([0.0, 0.0, 0.35])
perturbed_value = monomial_value(perturbed_P, CONIC_A) - monomial_value(perturbed_P, CONIC_B)
checks["conic_expression_on_conic_abs"] = float(abs(conic_value))
checks["conic_expression_after_perturb_abs"] = float(abs(perturbed_value))

Gproof = nx.DiGraph()
Gproof.add_edges_from([
    ("two balanced bracket monomials", "2592 raw determinant products"),
    ("2592 raw determinant products", "936 opposite cancellation pairs"),
    ("936 opposite cancellation pairs", "720 surviving coordinate terms"),
    ("720 surviving coordinate terms", "six-point conic test"),
    ("repeat rule", "expansion map sends relations to zero"),
    ("alternating rule", "expansion map sends relations to zero"),
    ("Grassmann-Pluecker rule", "expansion map sends relations to zero"),
    ("expansion map sends relations to zero", "equivalent bracket polynomials have same coordinate expansion"),
    ("equivalent bracket polynomials have same coordinate expansion", "six-point conic test"),
])
checks["proof_graph_reaches_conic_test"] = nx.has_path(Gproof, "Grassmann-Pluecker rule", "six-point conic test")

pos = {
    "two balanced bracket monomials": (0, 1.3),
    "2592 raw determinant products": (1.6, 1.3),
    "936 opposite cancellation pairs": (3.2, 1.3),
    "720 surviving coordinate terms": (4.8, 1.3),
    "six-point conic test": (6.4, 0.35),
    "repeat rule": (0, -0.4),
    "alternating rule": (0, -1.0),
    "Grassmann-Pluecker rule": (0, -1.6),
    "expansion map sends relations to zero": (2.4, -1.0),
    "equivalent bracket polynomials have same coordinate expansion": (4.8, -1.0),
}
fig, ax = plt.subplots(figsize=(11.8, 5.4))
node_colors = ["#dbeafe" if "rule" not in n else "#dcfce7" for n in Gproof.nodes]
nx.draw_networkx_nodes(Gproof, pos, node_size=2300, node_color=node_colors, edgecolors="#111827", linewidths=0.8, ax=ax)
nx.draw_networkx_edges(Gproof, pos, arrows=True, arrowstyle="-|>", arrowsize=14, edge_color="#374151", width=1.2, ax=ax)
nx.draw_networkx_labels(Gproof, pos, font_size=8.5, ax=ax)
ax.set_title("Bracket polynomial cancellation and proof-state graph")
ax.axis("off")
artifact_paths["cancellation_graph"] = save_current_figure(FIGURES / "bracket-polynomial-cancellation-proof-graph.png")

symbolic_summary = {
    "repeat_identity_zero": checks["symbolic_repeat_zero"],
    "alternation_identity_zero": checks["symbolic_alternation_zero"],
    "grassmann_pluecker_identity_zero": checks["symbolic_gp_zero"],
    "raw_coordinate_terms_before_collection": raw_coordinate_terms,
    "surviving_coordinate_terms_after_collection": surviving_terms,
    "cancelled_raw_terms": cancelled_terms,
    "conic_expression_on_sampled_conic_abs": checks["conic_expression_on_conic_abs"],
    "conic_expression_after_perturb_abs": checks["conic_expression_after_perturb_abs"],
}
artifact_paths["symbolic_checks"] = save_json(symbolic_summary, ARTIFACT_ROOT, "checks", "symbolic-bracket-checks.json")
symbolic_summary


## Applied Lab: Break A Weight And Watch The Invariant Fail

The Plotly lab varies the scale of one homogeneous column. A balanced cross-ratio quotient stays constant because that point has the same weight above and below the fraction. An intentionally unbalanced expression changes with the scale. This is the practical test for whether a bracket rational function is projectively meaningful.

In [ ]:
lambdas = np.linspace(0.25, 4.0, 80)
balanced_values = []
unbalanced_values = []
raw_num_values = []
raw_den_values = []
for lam in lambdas:
    D = np.eye(6)
    D[2, 2] = lam
    moved = line_config @ D
    num = monomial_value(moved, CROSS_NUM)
    den = monomial_value(moved, CROSS_DEN)
    raw_num_values.append(num)
    raw_den_values.append(den)
    balanced_values.append(num / den)
    unbalanced_values.append(det_bracket(moved, (1, 3, 5)) / det_bracket(moved, (1, 4, 5)))

balanced_values = np.array(balanced_values, dtype=float)
unbalanced_values = np.array(unbalanced_values, dtype=float)
checks["scaling_lab_balanced_range"] = float(balanced_values.max() - balanced_values.min())
checks["scaling_lab_unbalanced_range"] = float(unbalanced_values.max() - unbalanced_values.min())

fig_lab = go.Figure()
fig_lab.add_trace(go.Scatter(x=lambdas, y=balanced_values, mode="lines", name="balanced bracket quotient"))
fig_lab.add_trace(go.Scatter(x=lambdas, y=unbalanced_values, mode="lines", name="unbalanced bracket ratio"))
fig_lab.add_trace(go.Scatter(x=lambdas, y=np.array(raw_num_values) / raw_num_values[0] * balanced_values[0], mode="lines", name="numerator scale, normalized", line=dict(dash="dot")))
fig_lab.add_trace(go.Scatter(x=lambdas, y=np.array(raw_den_values) / raw_den_values[0] * balanced_values[0], mode="lines", name="denominator scale, normalized", line=dict(dash="dot")))
fig_lab.update_layout(
    title="Bracket quotient under one column rescaling",
    xaxis_title="scale applied to point 3",
    yaxis_title="value",
    template="plotly_white",
    height=500,
)
artifact_paths["scaling_lab"] = HTML / "bracket-weight-invariant-lab.html"
fig_lab.write_html(artifact_paths["scaling_lab"], include_plotlyjs=True, full_html=True)

{"balanced_range": checks["scaling_lab_balanced_range"], "unbalanced_range": checks["scaling_lab_unbalanced_range"], "html": as_book_path(artifact_paths["scaling_lab"])}


## Artifact Gallery

The main static artifacts are linked directly so the notebook remains readable before execution.

![Points to determinants Gamma map](../../artifacts/chapter-07-more-on-bracket-algebra/figures/points-to-determinants-gamma-map.png)

![Basis reconstruction from brackets](../../artifacts/chapter-07-more-on-bracket-algebra/figures/basis-reconstruction-from-brackets.png)

![Monomial weight balance](../../artifacts/chapter-07-more-on-bracket-algebra/figures/monomial-weight-balance.png)

![Bracket polynomial cancellation proof graph](../../artifacts/chapter-07-more-on-bracket-algebra/figures/bracket-polynomial-cancellation-proof-graph.png)

Open the interactive lab: [bracket-weight-invariant-lab.html](../../artifacts/chapter-07-more-on-bracket-algebra/html/bracket-weight-invariant-lab.html). The data ledgers are saved as `gamma-determinant-vector.csv`, `basis-reconstruction-ledger.csv`, and `monomial-weight-ledger.csv`.

In [ ]:
display_paths = [
    artifact_paths["gamma_map"],
    artifact_paths["basis_reconstruction"],
    artifact_paths["weight_balance"],
    artifact_paths["cancellation_graph"],
    artifact_paths["scaling_lab"],
    artifact_paths["gamma_table"],
    artifact_paths["reconstruction_table"],
    artifact_paths["weight_table"],
    artifact_paths["symbolic_checks"],
]
assert_artifacts(display_paths)
for item in display_paths[:4]:
    display_artifact(item, width=760)
display_artifact(display_paths[4], width="100%", height=520)
for item in display_paths[5:]:
    display_artifact(item)


## Takeaways

- `Gamma(P)` packages all plane brackets into one determinant vector; a projective transformation changes it by a single common determinant factor.
- If one bracket is nonzero, normalized bracket data reconstructs a representative matrix with the first three columns fixed as a basis.
- Grassmann-Pluecker relations are not optional identities; they are the consistency rules that make arbitrary bracket numbers behave like determinants.
- A rational bracket expression is projectively invariant when the numerator and denominator have the same total bracket degree and the same weight for every point label.
- Bracket algebra compresses coordinate algebra: short bracket polynomials can expand into many coordinate terms, and the meaningful proof work is often in the cancellations and rewrite rules.

In [ ]:
assert checks["gamma_common_scale_residual"] < 1e-10
assert checks["numeric_gp_residual"] < 1e-10
assert checks["basis_reconstruction_max_residual"] < 1e-10
assert checks["cross_ratio_error"] < 1e-10
assert checks["cross_ratio_weights_balanced"]
assert checks["conic_monomial_weights_balanced"]
assert checks["symbolic_repeat_zero"]
assert checks["symbolic_alternation_zero"]
assert checks["symbolic_gp_zero"]
assert checks["conic_raw_coordinate_terms"] == 2592
assert checks["conic_surviving_coordinate_terms"] == 720
assert checks["conic_cancelled_coordinate_terms"] == 1872
assert checks["conic_expression_on_conic_abs"] < 1e-8
assert checks["conic_expression_after_perturb_abs"] > 1e-3
assert checks["proof_graph_reaches_conic_test"]
assert checks["scaling_lab_balanced_range"] < 1e-12
assert checks["scaling_lab_unbalanced_range"] > 0.1

assert_artifacts(display_paths)
raster_stats = [image_stats(path) for path in display_paths if path.suffix.lower() == ".png"]
for stat in raster_stats:
    assert stat["width"] >= 300 and stat["height"] >= 240
    assert stat["pixel_std"] > 1.0

visual_checks = {
    "all_files_exist": all(path.exists() for path in display_paths),
    "cross_ratio_error": checks["cross_ratio_error"],
    "numeric_checks": checks,
    "raster_stats": [
        {**stat, "path": as_book_path(Path(stat["path"]))}
        for stat in raster_stats
    ],
    "display_artifacts": [as_book_path(path) for path in display_paths],
}
artifact_paths["visual_checks"] = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")

final = {
    "chapter": 7,
    "title": "More on Bracket Algebra",
    "source_span": "printed pages 109-126 / PDF pages 131-148",
    "artifacts": [as_book_path(path) for path in display_paths],
    "checks": checks,
    "notebook_executed": True,
}
artifact_paths["final_sanity"] = save_json(final, ARTIFACT_ROOT, "checks", "final-sanity.json")
final
